In [22]:
import os
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

In [ ]:
pd.set_option("display.expand_frame_repr", False)  # запрещаем перенос строк
import wquantiles  # библиотека для вычисления взвешенных статистик
from tqdm import tqdm

In [ ]:
adata_new = sc.read_h5ad("./data/adata_all_harm_ribo.h5ad")
# contains raw

In [5]:
# число экспрессируемых генов в каждой клетке
n_expr_genes_per_cell = adata_new.X.getnnz(axis=1)

# фильтруем клетки, где экспрессируется >= 500 генов
cells_to_keep = n_expr_genes_per_cell >= 500

# оставляем только эти клетки
adata = adata_new[cells_to_keep].copy()

In [7]:
adata

AnnData object with n_obs × n_vars = 1372418 × 43087
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final', 'Age_bin'
    uns: 'Tissue_global_colors'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

In [ ]:
def ct_passes_filter(sub):
    age_bins = pd.to_numeric(sub.obs["Age_bin"], errors="coerce").dropna().astype(int)
    if len(age_bins) == 0:
        return False

    n_cells = sub.obs["Age_bin"].value_counts()

    # условие 1: ≥3 интервала с ≥100 клеток
    cond1 = (n_cells >= 100).sum() >= 3

    # условие 2: интервал ≤31 и интервал ≥61 с ≥100 клеток
    cond2 = (n_cells.get(31, 0) >= 100) and (n_cells.get(61, 0) >= 100)

    # условие 3: интервал ≥51 с ≥100 клеток
    cond3 = (n_cells[n_cells.index >= 51] >= 100).any()

    # возвращаем True, если (cond1 или cond2) И cond3
    return (cond1 or cond2) and cond3

### Downsampling

In [9]:
import scipy.sparse as sp
import warnings

In [10]:
shared_cell_frac = 0.7
seed = 69

rng = np.random.default_rng(seed)


# Гарантируем CSR-формат
if sp.issparse(adata.X):
    if not sp.isspmatrix_csr(adata.X):
        adata.X = adata.X.tocsr()
else:
    raise TypeError("adata.X is not a sparse matrix")

In [ ]:
"""filtering & weighted downsampling of genes by cell types"""

filtered_adatas = []

for t in tqdm(sorted(adata.obs.Tissue_global.unique()), desc="Tissues"):
    adata_t = adata[adata.obs.Tissue_global == t].copy()
    for ct in tqdm(
        sorted(adata_t.obs.cell_type_final.unique()), desc=f"{t}", leave=False
    ):
        adata_ct = adata_t[adata_t.obs.cell_type_final == ct].copy()
        if not ct_passes_filter(adata_ct):
            continue

        # делаем DataFrame для удобства
        df = pd.DataFrame(
            {
                "n_expr_genes": adata_ct.X.getnnz(axis=1),
                "GSE_id": adata_ct.obs["GSE_id"].values,
                "GSM_id": adata_ct.obs["GSM_id"].values,
            }
        )

        # группируем по GSE_id и сразу фильтруем
        gse_stats = df.groupby("GSE_id", observed=True).agg(
            n_unique_gsm=("GSM_id", "nunique"),
            n_cells=("n_expr_genes", "size"),
            q75=("n_expr_genes", lambda x: x.quantile(0.75)),
        )

        # фильтр: >= 100 клеток
        gse_stats = gse_stats[gse_stats["n_cells"] >= 100]
        if gse_stats.empty:
            tqdm.write(f"No valid GSE groups after filtering for {t}, {ct}")
            continue

        # вычисляем вес и target_val
        gse_stats["weight"] = np.cbrt(gse_stats["n_cells"]) * gse_stats["n_unique_gsm"]
        gse_stats = gse_stats.sort_values(by="weight", ascending=False)

        w = wquantiles.median(gse_stats["q75"].values, gse_stats["weight"].values)
        target_val = int(np.ceil(w))

        # фильтруем adata_ct по GSE_id, которые прошли фильтр
        valid_gse_ids = set(gse_stats.index)
        adata_ct = adata_ct[adata_ct.obs["GSE_id"].isin(valid_gse_ids)].copy()

        if adata_ct.n_obs < 100:
            tqdm.write(f"All cells filtered out for {t}, {ct}: less than 100 cells")
            continue

        # доля клеток, в которых ген экспрессируется, и список генов
        gene_frac = adata_ct.X.getnnz(axis=0) / adata_ct.n_obs
        shared_genes_list = adata_ct.var_names[gene_frac >= shared_cell_frac]
        shared_gene_idx = set(adata_ct.var_names.get_indexer(shared_genes_list))
        tqdm.write(
            f"[{t} / {ct}] shared_genes={len(shared_genes_list)}", file=sys.stderr
        )
        # цикл по клеткам
        for i in range(adata_ct.n_obs):
            expressed_idx = adata_ct.X[i].indices
            shared_idx = [g for g in expressed_idx if g in shared_gene_idx]
            other_idx = [g for g in expressed_idx if g not in shared_gene_idx]
            n_needed = target_val - len(shared_idx)

            if n_needed <= 0:
                kept_other = []
            else:
                if len(other_idx) > n_needed:
                    kept_other = rng.choice(other_idx, size=n_needed, replace=False)
                else:
                    kept_other = other_idx

            keep_genes = set(shared_idx) | set(kept_other)
            keep_mask = np.isin(expressed_idx, list(keep_genes))
            adata_ct.X[i].data[~keep_mask] = 0

        adata_ct.X.eliminate_zeros()
        filtered_adatas.append(adata_ct)

In [15]:
final_adata = sc.concat(filtered_adatas, index_unique=None)

In [17]:
adata

AnnData object with n_obs × n_vars = 1372418 × 43087
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final', 'Age_bin'
    uns: 'Tissue_global_colors'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

In [20]:
final_adata

AnnData object with n_obs × n_vars = 1329794 × 43087
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final', 'Age_bin'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

In [ ]:
clean_adata = ad.AnnData(obs=final_adata.obs, X=final_adata.X, var=final_adata.var)
# тут прямые ссылки, с новым объектом не работать!!!
# либо сделать .copy()

In [ ]:
clean_adata.write(
    "./data/senepy_denovo_signatures_code/6.3_adata_ribo_downsampled.h5ad"
)